# 01 · Computational graphs: values forwards, gradients backwards

*Build Your Own Autograd: a Tiny Automatic Differentiation Engine*, notebook 1 of 4

In *Neural Networks from Scratch* you derived the backward pass of every layer by hand. Frameworks such as PyTorch do not work that way: they record each operation as your code runs and then differentiate that record automatically. In this course you build such an engine yourself, one typed piece at a time, and finish by training a small neural network with it.

This first notebook does everything by hand on a small expression stored as a Python dictionary, so that you can see every number. Notebooks 02 and 03 then turn the same ideas into an engine that records the graph for you.

## What you will build

- `forward_values(graph, inputs)`: evaluates every node of a graph, in order;
- `local_derivatives(op, args, out)`: the derivative of one operation with respect to each of its arguments;
- `backward_gradients(graph, values, output)`: the chain rule as a single backwards walk, giving the derivative of the output with respect to every node;
- `numerical_gradient(f, inputs)`: central differences, which check every gradient without using any derivative rule.

**Learning objectives.** By the end you can:

1. write an expression as a computational graph and read off an order in which to evaluate it;
2. work out the local derivative of each operation by hand;
3. apply the chain rule as a backwards walk, adding the contributions when a value is used more than once;
4. check gradients against central differences and choose a sensible step size.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 0:** the Python standard library only. The core cells use `math` and `random` (imported in the setup cell) and `copy`, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- One optional cell, tagged `visualization`, draws the graph with Matplotlib if it is installed. Nothing depends on it, so you can skip it.
- Nothing is downloaded, and nothing in this notebook is random.

## Setup

Run this cell first. It creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference. It also imports `copy`, which the next helper uses.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import math
import random

rng = random.Random(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    if isinstance(want, (int, float)) and isinstance(got, (int, float)):
        return math.isclose(got, want, rel_tol=1e-9, abs_tol=atol)
    return got == want


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-9):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import copy

### One more checking helper: `practice_with`

The functions in this notebook take dictionaries. A function that quietly changed a dictionary it was given would cause confusing errors much later, so the checks run your function through a **probe** first. `on_copies` is the probe used here: it hands your function deep copies of the test inputs and reports, next to your result, whether your function changed them. `practice_with(probe, name, yours, reference, cases)` then runs `practice` on the probed functions and, exactly like `practice`, gives back your own function once it passes and the reference otherwise.

In [ ]:
def practice_with(probe, name, yours, reference, cases, atol=1e-9):
    """Run practice on probe(yours) and probe(reference); return the function to use from now on.

    A probe wraps a function so that it returns plain numbers, lists or dicts
    that practice can compare. You get back your own function once it passes,
    otherwise the reference, exactly as with practice.
    """
    your_probe, reference_probe = probe(yours), probe(reference)
    chosen = practice(name, your_probe, reference_probe, cases, atol=atol)
    return yours if chosen is your_probe else reference


def on_copies(function):
    """Probe: call function on deep copies of the arguments and report whether it changed them."""
    def run(*args):
        copies = copy.deepcopy(args)
        result = function(*copies)
        return {"arguments unchanged": copies == args, "result": result}
    return run

## 1 · An expression is a graph

Take the squared error of one tanh neuron on one example,

$$L = \big(\tanh(w x + b) - t\big)^2 ,$$

with weight $w$, input $x$, bias $b$ and target $t$. A computer evaluates it one operation at a time, and every intermediate result can be given a name:

| node | operation | arguments | meaning |
|---|---|---|---|
| $p$ | `mul` | $w$, $x$ | the weighted input, $p = w x$ |
| $z$ | `add` | $p$, $b$ | the pre-activation, $z = p + b$ |
| $y$ | `tanh` | $z$ | the neuron's output, $y = \tanh z$ |
| $e$ | `sub` | $y$, $t$ | the error, $e = y - t$ |
| $L$ | `mul` | $e$, $e$ | the squared error, $L = e \cdot e$ |

Draw every value as a node and an arrow from each argument to the node that uses it, and you have a **computational graph**: a directed graph without cycles. The inputs $w$, $x$, $b$ and $t$ are its leaves, and $L$ is its output. Notice that $e$ is used **twice**, by $L = e \cdot e$; that will matter for the gradients.

We store the graph as a dictionary that maps each node's name to `(op, [argument names])`. Python dictionaries keep their insertion order, and every node is listed after its arguments, so reading the dictionary from top to bottom gives a valid order in which to evaluate it. Every value is a plain Python float: there are no arrays or shapes in this course, only single numbers.

In [ ]:
graph = {
    "p": ("mul", ["w", "x"]),
    "z": ("add", ["p", "b"]),
    "y": ("tanh", ["z"]),
    "e": ("sub", ["y", "t"]),
    "L": ("mul", ["e", "e"]),
}
inputs = {"w": 0.5, "x": 2.0, "b": -0.3, "t": 1.0}

for name, (op, args) in graph.items():
    print(f"{name} = {op}({', '.join(args)})")

## 2 · The forward pass

To evaluate the graph, visit the nodes in order and apply each operation to the values of its arguments. Those values are always known already, because every argument is either an input or a node listed earlier. `OPS` maps each operation's name to an ordinary Python function of floats.

The forward pass **keeps every intermediate value**, not just the output: the backward pass will need them.

In [ ]:
OPS = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
    "tanh": math.tanh,
}

p_value = OPS["mul"](inputs["w"], inputs["x"])   # the first node, by hand
print("p =", p_value)

### ✍️ Your turn: `forward_values`

Write `forward_values(graph, inputs)`:

- **inputs:** `graph`, a dictionary mapping each node's name to `(op, args)`, listed so that every node comes after its arguments; `inputs`, a dictionary mapping each leaf's name to a float;
- **output:** a **new** dictionary holding the value of every input and every node, for example `{"w": 0.5, …, "L": 0.1565…}`.

Hints: start from a copy, `values = dict(inputs)`, so that the caller's dictionary is left alone. Then, for each `name, (op, args)` in `graph.items()`, look up the arguments' values and call `OPS[op](*argument_values)`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def forward_values_reference(graph, inputs):
    values = dict(inputs)
    for name, (op, args) in graph.items():
        values[name] = OPS[op](*[values[arg] for arg in args])
    return values

In [ ]:
def forward_values(graph, inputs):
    """Return a new dict with the value of every input and every node of graph."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
forward_values = practice_with(on_copies, "forward_values", forward_values, forward_values_reference, cases=[
    (graph, inputs),
    ({"s": ("mul", ["a", "a"]), "r": ("sub", ["s", "a"])}, {"a": 3.0}),
    ({}, {"a": 1.5}),                       # edge case: no operations, only an input
])
values = forward_values(graph, inputs)
for name, value in values.items():
    print(f"{name:>2} = {value:+.6f}")

**Expected behaviour.** ✅, then the four inputs followed by $p = 1$, $z = 0.7$, $y = \tanh 0.7 \approx 0.604368$, $e \approx -0.395632$ and $L \approx 0.156525$. The forward pass is ordinary arithmetic; the only new idea is that every intermediate value is kept, under its node's name.

## 3 · Drawing the graph

`draw_graph(graph, output, values, grads)` prints the graph as a tree. It starts at the output and indents each node's arguments beneath it, so you read it from the bottom up to follow the forward pass and from the top down to follow the backward pass. A node that has already been drawn is shown again as one line marked "(again)". The `grads` argument is optional; you will fill it in in section 5.

In [ ]:
def draw_graph(graph, output, values, grads=None):
    """Print the graph as a tree rooted at output: name, operation, value and, if given, gradient."""
    drawn = set()

    def describe(name):
        how = graph[name][0] if name in graph else "input"
        text = f"{name} [{how}] = {values[name]:+.4f}"
        if grads is not None:
            text += f"   grad {grads[name]:+.4f}"
        return text

    def walk(name, prefix, last, top):
        branch = "" if top else ("└─ " if last else "├─ ")
        again = name in drawn and name in graph
        print(prefix + branch + describe(name) + ("   (again)" if again else ""))
        if again or name not in graph:
            return
        drawn.add(name)
        args = graph[name][1]
        inner = prefix if top else prefix + ("   " if last else "│  ")
        for index, arg in enumerate(args):
            walk(arg, inner, index == len(args) - 1, False)

    walk(output, "", True, True)


draw_graph(graph, "L", values)

**Optional picture.** The next cell is tagged `visualization`. It draws the same graph with Matplotlib, if Matplotlib is installed (it is in JupyterLite, on the course server and in Colab), placing each node one column to the right of its furthest argument. Nothing later depends on it.

In [ ]:
# Optional: needs Matplotlib. Nothing later in the notebook depends on this cell.
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("Matplotlib is not available; the text drawing above shows the same graph.")

if plt is not None:
    depth = {name: 0 for name in inputs}
    for name, (op, args) in graph.items():
        depth[name] = 1 + max(depth[arg] for arg in args)
    columns = {}
    for name, level in depth.items():
        columns.setdefault(level, []).append(name)
    position = {}
    for level, names in columns.items():
        for row, name in enumerate(names):
            position[name] = (level, (len(names) - 1) / 2 - row)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for name, (op, args) in graph.items():
        for arg in set(args):
            ax.annotate("", xy=position[name], xytext=position[arg],
                        arrowprops={"arrowstyle": "->", "color": "0.5", "shrinkA": 22, "shrinkB": 22})
    for name, (x_pos, y_pos) in position.items():
        how = graph[name][0] if name in graph else "input"
        ax.text(x_pos, y_pos, f"{name}\n{how}\n{values[name]:.3f}", ha="center", va="center",
                bbox={"boxstyle": "round", "facecolor": "white", "edgecolor": "0.3"})
    ax.set_xlim(-0.6, max(depth.values()) + 0.6)
    ax.set_ylim(-2.2, 2.2)
    ax.axis("off")
    ax.set_title("Each arrow points from an argument to the node that uses it (e feeds L twice)")
    plt.show()

## 4 · Local derivatives

A node knows only its own operation. Its **local derivative** with respect to one argument says how fast the node's value changes when that argument changes and everything else is held fixed. For our four operations:

| operation | node | with respect to the first argument | with respect to the second |
|---|---|---|---|
| `add` | $a + b$ | $1$ | $1$ |
| `sub` | $a - b$ | $1$ | $-1$ |
| `mul` | $a \cdot b$ | $b$ | $a$ |
| `tanh` | $\tanh a$ | $1 - \tanh^2 a = 1 - \text{out}^2$ | |

Two details are worth noticing. For `mul`, each argument's derivative is the **other** argument. For `tanh`, the derivative can be written using the node's own output, $\frac{d}{da}\tanh a = 1 - \tanh^2 a$; since the forward pass kept that output, the backward pass need not compute tanh again.

These numbers are *local*: they describe one node, and say nothing yet about $L$. The demo checks the tanh row with a central difference, which you will meet properly in section 6.

In [ ]:
def central_difference(f, x, h=1e-6):
    """Estimate f'(x) from two nearby values of f: (f(x + h) - f(x - h)) / (2h)."""
    return (f(x + h) - f(x - h)) / (2 * h)


z0 = 0.7
print("1 - tanh(z)**2     =", 1 - math.tanh(z0) ** 2)
print("central difference =", central_difference(math.tanh, z0))

### ✍️ Your turn: `local_derivatives`

Write `local_derivatives(op, args, out)`:

- **inputs:** `op`, one of `"add"`, `"sub"`, `"mul"` and `"tanh"`; `args`, the list of the node's argument values (two floats, or one for `tanh`); `out`, the node's own value from the forward pass;
- **output:** a list with one float per argument, the derivative of the node with respect to that argument, as in the table. For example, `local_derivatives("mul", [2.0, -3.0], -6.0)` returns `[-3.0, 2.0]`.

Hints: one `if` per operation. Use `out` for tanh. Raise a `ValueError` for an operation you do not know.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def local_derivatives_reference(op, args, out):
    if op == "add":
        return [1.0, 1.0]
    if op == "sub":
        return [1.0, -1.0]
    if op == "mul":
        return [args[1], args[0]]
    if op == "tanh":
        return [1.0 - out * out]
    raise ValueError(f"no derivative rule for the operation {op!r}")

In [ ]:
def local_derivatives(op, args, out):
    """Derivatives of one node with respect to each of its arguments, as a list."""
    # ✍️ Your code here (one short branch per operation)
    raise NotImplementedError

In [ ]:
local_derivatives = practice_with(on_copies, "local_derivatives", local_derivatives, local_derivatives_reference, cases=[
    ("add", [2.0, 3.0], 5.0),
    ("sub", [1.0, 4.0], -3.0),
    ("mul", [2.0, -3.0], -6.0),
    ("tanh", [0.7], math.tanh(0.7)),
])
print("local derivatives of L = e·e, one per use of e:", local_derivatives("mul", [values["e"], values["e"]], values["L"]))

**Expected behaviour.** ✅ on all four operations. The last line shows that $L = e \cdot e$ has two local derivatives, one for each argument, and both equal $e \approx -0.3956$: each use of $e$ sees the other use as "the other argument".

## 5 · The chain rule as a backwards walk

Now the global question: how does $L$ change when $w$ changes? Changing $w$ changes $p$, which changes $z$, then $y$, then $e$ and finally $L$. The chain rule multiplies the local derivatives along the way:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial e}\,\frac{\partial e}{\partial y}\,\frac{\partial y}{\partial z}\,\frac{\partial z}{\partial p}\,\frac{\partial p}{\partial w} .$$

Rather than writing such a product for every input, walk the graph backwards **once**. Write $\bar v = \partial L / \partial v$ for the gradient of the output with respect to node $v$. Start with $\bar L = 1$, then visit the nodes in reverse order. Each node passes its own gradient to its arguments, multiplied by the local derivative:

$$\bar a \mathrel{+}= \bar v \cdot \frac{\partial v}{\partial a} \qquad \text{for every argument } a \text{ of } v .$$

**The `+=` matters.** If a value is used in several places, $L$ depends on it along several paths, and the chain rule *adds* their contributions. Here $e$ is used twice by $L = e \cdot e$: each use contributes $\bar L \cdot e$, so $\bar e = 2e$, which is exactly the derivative of $e^2$. And because the walk goes in reverse order, a node passes on its gradient only after every node that uses it has finished adding to it.

By hand, with the values from the forward pass:

| gradient | rule | value |
|---|---|---|
| $\bar L$ | the seed | $1$ |
| $\bar e$ | $\bar L \cdot e + \bar L \cdot e$ | $2 \times (-0.395632) = -0.791264$ |
| $\bar y$ | $\bar e \cdot 1$ | $-0.791264$ |
| $\bar t$ | $\bar e \cdot (-1)$ | $+0.791264$ |
| $\bar z$ | $\bar y \cdot (1 - y^2)$ | $-0.791264 \times 0.634740 = -0.502247$ |
| $\bar p$ and $\bar b$ | $\bar z \cdot 1$ | $-0.502247$ |
| $\bar w$ | $\bar p \cdot x$ | $-0.502247 \times 2 = -1.004494$ |
| $\bar x$ | $\bar p \cdot w$ | $-0.502247 \times 0.5 = -0.251123$ |

The signs make sense. The output $y \approx 0.60$ is below the target $t = 1$, so increasing $w$, $x$ or $b$ (which pushes $y$ up) lowers the loss, while increasing $t$ raises it. The demo performs the first step of the walk, at node $L$.

In [ ]:
grads_by_hand = {name: 0.0 for name in values}
grads_by_hand["L"] = 1.0
for arg, local in zip(["e", "e"], local_derivatives("mul", [values["e"], values["e"]], values["L"])):
    grads_by_hand[arg] += grads_by_hand["L"] * local     # e receives two contributions
print("dL/de after visiting L:", grads_by_hand["e"], "  and 2e =", 2 * values["e"])

### ✍️ Your turn: `backward_gradients`

Write `backward_gradients(graph, values, output)`:

- **inputs:** `graph` as before; `values`, the dictionary returned by `forward_values`; `output`, the name of the node to differentiate (here `"L"`);
- **output:** a new dictionary mapping every name in `values` to the derivative of the output with respect to it.

The steps: (1) start every gradient at `0.0` and set the output's to `1.0`; (2) visit the node names in **reverse** order, `reversed(list(graph))`; (3) for each node, compute `local_derivatives(op, argument_values, values[name])` and add `grads[name] * local` to each argument's gradient.

Hints: `zip(args, locals)` pairs each argument with its local derivative. Always use `+=`, never `=`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def backward_gradients_reference(graph, values, output):
    grads = {name: 0.0 for name in values}
    grads[output] = 1.0
    for name in reversed(list(graph)):
        op, args = graph[name]
        local = local_derivatives_reference(op, [values[arg] for arg in args], values[name])
        for arg, derivative in zip(args, local):
            grads[arg] += grads[name] * derivative
    return grads

In [ ]:
def backward_gradients(graph, values, output):
    """Return a new dict: d(output)/d(name) for every name in values."""
    # ✍️ Your code here (about seven lines)
    raise NotImplementedError

In [ ]:
square_graph = {"s": ("mul", ["a", "a"]), "r": ("sub", ["s", "a"])}   # r = a·a - a: a is used three times
square_values = forward_values(square_graph, {"a": 3.0})
backward_gradients = practice_with(on_copies, "backward_gradients", backward_gradients, backward_gradients_reference, cases=[
    (graph, values, "L"),
    (square_graph, square_values, "r"),
    (graph, values, "z"),                   # edge case: the gradient of an intermediate node
])
grads = backward_gradients(graph, values, "L")
draw_graph(graph, "L", values, grads)

**Expected behaviour.** ✅ on all three cases, and the drawing now shows each node's gradient: exactly the numbers in the table above, from $\bar L = 1$ down to $\bar w \approx -1.0045$ and $\bar x \approx -0.2511$. For `square_graph`, $r = a^2 - a$ at $a = 3$ gives $\partial r / \partial a = 2a - 1 = 5$: three contributions ($3 + 3 - 1$) added into one gradient. The third case asks for the gradient of $z$ instead of $L$: the nodes above $z$ keep a gradient of 0 and pass nothing on.

## 6 · Check every gradient numerically

How do we know the backward walk is right? Compare it with a method that uses no derivative rules at all: nudge one input up and down by a small step $h$, rerun the forward pass each time, and measure the slope,

$$\frac{\partial L}{\partial w} \approx \frac{L(w + h) - L(w - h)}{2h} .$$

This **central difference** has an error proportional to $h^2$, so with $h = 10^{-6}$ it usually agrees with the true derivative to about nine or ten significant digits; rounding error stops it doing better. It costs two forward passes for **every** input, so it is far too slow for training, but it is ideal for checking.

In [ ]:
def loss_of(point):
    """L as a function of the four inputs: one full forward pass."""
    return forward_values(graph, point)["L"]


w_ahead, w_behind = dict(inputs, w=inputs["w"] + 1e-6), dict(inputs, w=inputs["w"] - 1e-6)
print("numerical dL/dw =", (loss_of(w_ahead) - loss_of(w_behind)) / 2e-6)
print("backward walk   =", grads["w"])

### ✍️ Your turn: `numerical_gradient`

Write `numerical_gradient(f, inputs, h=1e-6)`:

- **inputs:** `f`, a function that takes a dictionary of inputs and returns a float; `inputs`, a dictionary of floats; `h`, the step;
- **output:** a new dictionary mapping each input's name to $\big(f(\text{inputs with that one } + h) - f(\text{inputs with that one } - h)\big) / 2h$.

Hints: build a shifted copy for each evaluation, such as `ahead = dict(inputs)` followed by `ahead[name] += h`. Do not change `inputs` itself; the check reports it if you do.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def numerical_gradient_reference(f, inputs, h=1e-6):
    gradient = {}
    for name in inputs:
        ahead, behind = dict(inputs), dict(inputs)
        ahead[name] += h
        behind[name] -= h
        gradient[name] = (f(ahead) - f(behind)) / (2 * h)
    return gradient

In [ ]:
def numerical_gradient(f, inputs, h=1e-6):
    """Central-difference estimate of the derivative of f with respect to each input."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
def product(point):
    return point["a"] * point["b"]


def wave(point):
    return math.sin(3.0 * point["u"]) + point["u"] ** 2


numerical_gradient = practice_with(on_copies, "numerical_gradient", numerical_gradient, numerical_gradient_reference, cases=[
    (product, {"a": 2.0, "b": -3.0}),
    (wave, {"u": 0.3}),
    (loss_of, inputs),
])
numerical = numerical_gradient(loss_of, inputs)
print(f"{'input':>5} {'backward walk':>15} {'central difference':>19} {'gap':>9}")
for name in inputs:
    gap = abs(grads[name] - numerical[name])
    print(f"{name:>5} {grads[name]:>15.10f} {numerical[name]:>19.10f} {gap:>9.1e}")

**Expected behaviour.** ✅, then a table in which the two columns agree to about ten decimal places for all four inputs, with gaps of $10^{-10}$ or less. Two completely different methods give the same numbers, which is strong evidence that both are right. Notice the cost: four inputs needed eight forward passes, while the backward walk needed one forward pass and one backward walk.

## 🧪 Experiments

### 🧪 Try it: change the inputs

Change the numbers in the first line and run the cell again. Try `"w": 3.0` together with `"t": -1.0`: the neuron then confidently gives the wrong answer.

In [ ]:
try_inputs = {"w": 0.5, "x": 2.0, "b": -0.3, "t": 1.0}   # try "w": 3.0 with "t": -1.0, then run this cell again
try_values = forward_values(graph, try_inputs)
try_grads = backward_gradients(graph, try_values, "L")
print(f"y = {try_values['y']:+.6f}, L = {try_values['L']:.6f}")
for name in try_inputs:
    print(f"dL/d{name} = {try_grads[name]:+.6f}")

**What to notice.** With $w = 3$ and $t = -1$, $z = 5.7$ and $y \approx 0.99998$, so the loss is almost 4, about as wrong as this neuron can be. Yet $\partial L / \partial w$ is only about $3.6 \times 10^{-4}$, because $1 - y^2 \approx 4.5 \times 10^{-5}$ multiplies everything below the tanh. A **saturated** tanh passes almost no gradient back, however large the error: one reason why initialisation matters (notebook 04).

### 🧪 Try it: the step size $h$

In [ ]:
step = 1e-6   # try 1e-2, 1e-4, 1e-9 and 1e-12, then run this cell again
estimate = numerical_gradient(loss_of, inputs, h=step)
worst = max(abs(estimate[name] - grads[name]) for name in inputs)
print(f"h = {step:.0e}: largest gap to the backward walk = {worst:.1e}")

**What to notice.** A large step such as $10^{-2}$ gives a gap of about $4 \times 10^{-4}$: this is the truncation error, which shrinks like $h^2$, so each tenfold smaller step makes it about a hundred times smaller. A tiny step such as $10^{-12}$ is worse again, about $4 \times 10^{-5}$: each computed value of $L$ carries a rounding error of roughly $10^{-17}$, and dividing by $2h$ magnifies it. The gap is smallest near $h = 10^{-6}$; steps between $10^{-7}$ and $10^{-5}$ work well for central differences on numbers of this size.

## 7 · Put it together: a graph with a shared input

A second graph: a tiny network with two tanh units that read the **same** input $x$, followed by a squared error,

$$L = \big(\tanh(w_1 x) + v \tanh(w_2 x) - t\big)^2 .$$

Now $x$ reaches $L$ along two paths, one through each unit, so its gradient must be the sum of two contributions. The cell runs your forward pass, your backward walk and your numerical gradient, draws the result, and checks it all with `assert`s.

In [ ]:
net = {
    "a1": ("mul", ["w1", "x"]),
    "h1": ("tanh", ["a1"]),
    "a2": ("mul", ["w2", "x"]),
    "h2": ("tanh", ["a2"]),
    "q": ("mul", ["v", "h2"]),
    "y": ("add", ["h1", "q"]),
    "e": ("sub", ["y", "t"]),
    "L": ("mul", ["e", "e"]),
}
net_inputs = {"x": 0.8, "w1": 1.5, "w2": -0.7, "v": 2.0, "t": 0.5}

net_values = forward_values(net, net_inputs)
net_grads = backward_gradients(net, net_values, "L")
net_numerical = numerical_gradient(lambda point: forward_values(net, point)["L"], net_inputs)
draw_graph(net, "L", net_values, net_grads)

worst_gap = max(abs(net_grads[name] - net_numerical[name]) for name in net_inputs)
print(f"\nlargest gap between the backward walk and central differences: {worst_gap:.1e}")

# x reaches L along two paths, through a1 and through a2: its gradient is the sum of both.
through_a1 = net_grads["a1"] * net_values["w1"]
through_a2 = net_grads["a2"] * net_values["w2"]
print(f"dL/dx = {net_grads['x']:+.6f} = {through_a1:+.6f} (via a1) + {through_a2:+.6f} (via a2)")

assert net_values["L"] >= 0, "a squared error is never negative"
assert worst_gap < 1e-6, "the backward walk disagrees with central differences"
assert math.isclose(net_grads["x"], through_a1 + through_a2, rel_tol=1e-12), "dL/dx must be the sum of its two paths"
assert set(net_grads) == set(net_values), "every input and node should have a gradient"

**What to notice.** All four `assert`s pass. The two paths from $x$ pull in **opposite** directions here, about $-0.62$ through $a_1$ and $+1.42$ through $a_2$, and the gradient is their sum, about $+0.79$. Computing each path separately and adding would also work, but the number of paths can grow exponentially with depth; the backward walk visits each node once whatever the number of paths, because the gradient arriving at a node already sums every path above it.

## Exercises

### Exercise 1 · Division

Add a rule for `div`, the node $a / b$. Work out $\partial (a/b) / \partial a$ and $\partial (a/b) / \partial b$ on paper first, then write `div_local_derivatives(a, b)`, which returns the two derivatives as a list.

In [ ]:
def div_local_derivatives_reference(a, b):
    return [1.0 / b, -a / (b * b)]

In [ ]:
def div_local_derivatives(a, b):
    """[d(a/b)/da, d(a/b)/db] for numbers a and b, with b != 0."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
div_local_derivatives = practice("div_local_derivatives", div_local_derivatives, div_local_derivatives_reference, cases=[
    (6.0, 3.0),
    (1.0, -0.5),
    (0.0, 2.0),                             # a = 0: the quotient does not change with b
])
print("central differences:", [central_difference(lambda a: a / 3.0, 6.0), central_difference(lambda b: 6.0 / b, 3.0)])
print("your rule:          ", div_local_derivatives(6.0, 3.0))

### Exercise 2 · The gradient as a sum over paths (stretch)

The chain rule has a second reading: $\partial\,\text{output} / \partial\,\text{source}$ is the **sum over every path** from the source to the output of the product of the local derivatives along that path. Write `path_gradient(graph, values, output, source)`, which computes it directly by recursion:

- if `source == output`, return `1.0`;
- otherwise, for every node that uses `source` as an argument (once per use), add (that node's local derivative with respect to this use) × `path_gradient(graph, values, output, node)`.

It must agree with `backward_gradients`. Think about how many times it visits each node.

In [ ]:
def path_gradient_reference(graph, values, output, source):
    if source == output:
        return 1.0
    total = 0.0
    for name, (op, args) in graph.items():
        local = local_derivatives_reference(op, [values[arg] for arg in args], values[name])
        for arg, derivative in zip(args, local):
            if arg == source:
                total += derivative * path_gradient_reference(graph, values, output, name)
    return total

In [ ]:
def path_gradient(graph, values, output, source):
    """d(output)/d(source) as a sum over every path, by recursion."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
path_gradient = practice_with(on_copies, "path_gradient", path_gradient, path_gradient_reference, cases=[
    (graph, values, "L", "w"),
    (net, net_values, "L", "x"),
    (net, net_values, "L", "t"),
])
for name in net_inputs:
    print(f"dL/d{name:<2}: paths {path_gradient(net, net_values, 'L', name):+.8f}   backward walk {net_grads[name]:+.8f}")

### Exercise 3 · Open questions

1. `numerical_gradient` needs two forward passes per input. How many forward passes would it need for a model with one million parameters, and what does the backward walk need instead?
2. Picture a graph in which every node uses the previous one twice: $s_1 = s_0 \cdot s_0$, $s_2 = s_1 \cdot s_1$, and so on up to $s_{20}$. How many paths lead from $s_0$ to $s_{20}$? How many steps does `backward_gradients` take? What does this say about `path_gradient`?
3. Why does the tanh rule use the node's **output** rather than its input, and what must the forward pass therefore keep?

## Common errors and debugging

| Symptom | Likely cause | Fix |
|---|---|---|
| `KeyError` in `forward_values` | a node is listed before one of its arguments, or a name is misspelt | list every node after its arguments, and check the spelling |
| a value used twice gets half its gradient ($\bar e = e$ instead of $2e$) | `grads[arg] = …` overwrote the first contribution | always add: `grads[arg] += grads[name] * derivative` |
| every gradient is 0 | the output's gradient was never seeded | set `grads[output] = 1.0` before the walk |
| gradients near the output are right, earlier ones wrong | the walk went forwards, so nodes passed on their gradient before receiving it | iterate over `reversed(list(graph))` |
| the tanh gradient is slightly off | `1 - math.tanh(out) ** 2` applies tanh twice | use `1 - out ** 2`; `out` is already the tanh |
| a check reports `'arguments unchanged': False` | the function modified a dictionary it was given | copy first: `values = dict(inputs)`, `ahead = dict(inputs)` |
| numerical and backward gradients differ in the fifth digit | the step $h$ is too large, or far too small | use $h$ between $10^{-7}$ and $10^{-5}$ |

## Summary

- An expression is a **computational graph**: the leaves are inputs, and every other node applies one operation to earlier nodes. Listing each node after its arguments gives an evaluation order.
- The **forward pass** evaluates the nodes in that order and keeps every value.
- Each operation has a simple **local derivative**: `add` gives $(1, 1)$, `sub` $(1, -1)$, `mul` $(b, a)$ and `tanh` $1 - \text{out}^2$.
- The **backward walk** seeds the output with 1, visits the nodes in reverse and adds gradient × local derivative into every argument. Contributions add up whenever a value is used more than once.
- **Central differences** check any gradient using forward passes only, at a cost of two passes per input.

You built `forward_values`, `local_derivatives`, `backward_gradients` and `numerical_gradient`.

## Next

In **02 · A scalar autodiff engine** (`02_scalar_autodiff.ipynb`) the graph stops being a dictionary that you write in advance. You build a `Value` class whose arithmetic records the graph as it runs, and each operation carries its own backward rule as a small function, a closure.